In [5]:
import pandas as pd

from unibench.benchmarks_zoo.registry import list_benchmarks
from unibench.models_zoo.registry import list_models
from unibench.output import OutputHandler
import seaborn as sns
import matplotlib.pyplot as plt

# models = list_models('vllm') + ['siglip2_so400_16_512']
models = list_models('vllm') + ['siglip2_so400_16_256', 'clip_vitL14', 'eva01_vitG14_plus_2b' , 'siglip2_so400_16_384', 'siglip2_so400_16_512', 'eva01_vitG14_400m']
vllm_models = list_models('vllm')

benchmarks = list_benchmarks()

outputhandler = OutputHandler(output_dir='/home/haltahan6/unibench/outputs', download_all_precomputed=False)

outputhandler.load_all_csv(
    model_name=models,
    benchmark_name=benchmarks,
)

results = outputhandler.query(**{"benchmark_name": benchmarks, "model_name": models})

# Ensure image_name consistency between contrastive models and vllm models
contrastive_models = ['siglip2_so400_16_256', 'clip_vitL14', 'eva01_vitG14_plus_2b', 'siglip2_so400_16_384', 'siglip2_so400_16_512', 'eva01_vitG14_400m']

# Get unique image names from vllm models to use as reference
vllm_image_names = set(results[results['model_name'].isin([model_mappings.get(m, m) for m in vllm_models])]['image_name'].dropna())

# Filter contrastive models to only include image names that exist in vllm results
results = results[
    ~results['model_name'].isin([model_mappings.get(m, m) for m in contrastive_models]) |
    results['image_name'].isin(vllm_image_names)
]

from unibench.common_utils.utils import get_model_mappings
model_mappings = get_model_mappings('model_type')
results['model_type'] = results["model_name"].map(model_mappings)
model_mappings = get_model_mappings('name')
results['model_name'] = results["model_name"].map(model_mappings)

from unibench.common_utils.utils import get_benchmark_mappings
benchmark_mappings = get_benchmark_mappings("capability")
results["capability"] = results["benchmark_name"].map(benchmark_mappings)
benchmark_mappings = get_benchmark_mappings("benchmark_type")
results["benchmark_type"] = results["benchmark_name"].map(benchmark_mappings)
benchmark_mappings = get_benchmark_mappings("num_classes")
results["num_classes"] = results["benchmark_name"].map(benchmark_mappings)

results = results[(results['task_name'] == 'multi_choice_classification') | (results['model_name'].str.contains('CLIP')) | (results['model_name'].str.contains('EVA')) | (results['model_name'].str.contains('SigLIP')) | (results['task_name'] == 'multi_choice_relation')]

File not found:  /home/haltahan6/unibench/outputs/chameleon_30b/sugarcrepe.f
File not found:  /home/haltahan6/unibench/outputs/chameleon_30b/vg_attribution.f
File not found:  /home/haltahan6/unibench/outputs/chameleon_30b/vg_relation.f
File not found:  /home/haltahan6/unibench/outputs/chameleon_30b/winoground.f
File not found:  /home/haltahan6/unibench/outputs/gemma3_12b/bivlc.f
File not found:  /home/haltahan6/unibench/outputs/gemma3_12b/caltech101.f
File not found:  /home/haltahan6/unibench/outputs/gemma3_12b/cars.f
File not found:  /home/haltahan6/unibench/outputs/gemma3_12b/cifar10.f
File not found:  /home/haltahan6/unibench/outputs/gemma3_12b/cifar100.f
File not found:  /home/haltahan6/unibench/outputs/gemma3_12b/clevr_count.f
File not found:  /home/haltahan6/unibench/outputs/gemma3_12b/clevr_distance.f
File not found:  /home/haltahan6/unibench/outputs/gemma3_12b/coco_order.f
File not found:  /home/haltahan6/unibench/outputs/gemma3_12b/countbench.f
File not found:  /home/haltahan6

In [8]:
vllm_capability = ['relations', 'corruption', 'imagenet', 'specifies classification', 'counting', 'spatial understanding', 'pose detection', 'depth estimation', 'geographic diversity']
contrastive_capability = ['natural transformations', 'rendition', 'standard object recognition', 'challenging imagenet', 'satellite', 'texture detection', 'character recognition', 'medical', 'scene recognition']

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Prepare the data
# Filter for rows with prompts and capability labels
classifier_data = results[
    (results['prompt'].notna()) & 
    (results['capability'].notna())
].copy()

# Create binary target: 1 for vllm capabilities, 0 for contrastive capabilities
classifier_data['target'] = classifier_data['capability'].apply(
    lambda x: 1 if x in vllm_capability else 0
)

# Get unique prompt-capability pairs to avoid data leakage
unique_prompts = classifier_data[['prompt', 'target']].drop_duplicates()

print(f"Total unique prompts: {len(unique_prompts)}")
print(f"VLLM capability prompts: {(unique_prompts['target'] == 1).sum()}")
print(f"Contrastive capability prompts: {(unique_prompts['target'] == 0).sum()}")

# Start with 10% of data for training
X = unique_prompts['prompt'].values
y = unique_prompts['target'].values

# Split into train (10%) and test (90%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.9, random_state=42, stratify=y
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

# Simple TF-IDF + Logistic Regression classifier
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train classifier
classifier = LogisticRegression(random_state=42)
classifier.fit(X_train_tfidf, y_train)

# Evaluate
y_pred = classifier.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nAccuracy on held-out test set: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Contrastive', 'VLLM']))

Total unique prompts: 1380163
VLLM capability prompts: 502147
Contrastive capability prompts: 878016
Training set size: 138016
Test set size: 1242147

Accuracy on held-out test set: 0.8997

Classification Report:
              precision    recall  f1-score   support

 Contrastive       0.91      0.93      0.92    790215
        VLLM       0.88      0.84      0.86    451932

    accuracy                           0.90   1242147
   macro avg       0.89      0.89      0.89   1242147
weighted avg       0.90      0.90      0.90   1242147

